# s13 — Background Tasks with Notifications

**What this teaches:** how the agent fires long-running shell work without blocking the loop. The `bg.Queue` exposes a `bash_background` tool (start), a notification stream (completion), and helpers for listing/waiting.

**Concept anchor:** the agent loop is a request/response cycle — anything that takes minutes (builds, deploys, syncs) would stall the model. `bg.Queue` is a side channel: the tool returns immediately with a job ID, the goroutine runs the command, and a notification lands when it's done.

## Prerequisites

- LLM provider configured. Default is `openai_compat` for self-hosted Ollama / vLLM — see [docs/providers.md](../../docs/providers.md).

## 1. Imports

`internal/bg` provides the queue; `tool` is the ADK tool interface we use to pass `q.Tool()` to the agent.

In [ ]:
import (
    "context"
    "fmt"
    "os"
    "time"

    "google.golang.org/adk/tool"

    "github.com/blouargant/omnis/core/agentkit"
    "github.com/blouargant/omnis/core/stream"
    "github.com/blouargant/omnis/internal/bg"
)

## 2. Helper

Notebook-safe `must` (panic, not `os.Exit`).

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. Build the background queue

`bg.NewQueue(16)` creates a queue with a notification buffer of 16. When a background command finishes, a `Notification` is pushed onto the queue — `Drain()` returns all pending ones at once.

In [ ]:
ctx := context.Background()
llm, err := agentkit.NewModel(ctx)
must(err)

q := bg.NewQueue(16)
fmt.Println("bg queue ready")

## 4. Build the agent

Only one tool is wired here: `q.Tool()` is the `bash_background` tool. The instruction tells the model to use it and let the user know completion will arrive out-of-band.

In [ ]:
a, err := agentkit.New(agentkit.AgentConfig{
    Name:        "s13_bg",
    Description: "Background-task demo.",
    Instruction: "When asked, start a background task with bash_background and report you'll be notified when done.",
    Model:       llm,
    Tools:       []tool.Tool{q.Tool()},
})
must(err)
r, err := agentkit.Runner("s13", a)
must(err)

## 5. Run the agent

The agent calls `bash_background` and returns immediately — *the model does not wait* for the sleep to finish.

In [ ]:
prompt := "Start a background task that runs `sleep 1 && echo done`."
must(stream.Print(os.Stdout, agentkit.RunOnce(ctx, r, prompt)))

## 6. Drain notifications

Now we poll `q.Drain()` until the deadline passes — this is what the **host** application (server, TUI, console) is responsible for. In the real harness the drain loop runs concurrently with the model loop; the model can be asked again later and learn about completions then.

In [ ]:
deadline := time.After(3 * time.Second)
for {
    done := false
    select {
    case <-deadline:
        done = true
    default:
    }
    if done {
        break
    }
    for _, n := range q.Drain() {
        fmt.Println(bg.FormatNotification(n))
    }
    time.Sleep(200 * time.Millisecond)
}

## What to look for

- The model's response arrives in a second or two; the notification lands ~1 second later, after `sleep 1` finishes.
- Compare to [s05_tools](../s05_tools/s05_tools.ipynb)'s synchronous `bash` — that one blocks until the command exits.
- The notification format is decided by `bg.FormatNotification` — the host UI can choose to surface this to the user as a toast, a chat-style message, etc.

## Try it yourself

1. Replace `sleep 1` with `sleep 10` and start the agent flow. Re-run the model in a second cell while the job is still running, asking "is my background job done?" — by then the queue might still be empty.
2. Add `bg_list` and `bg_wait` to the toolset (also exposed from `q`) so the agent itself can poll.